# Creating the vector store

## There are several ways of creating the configuration
```
_INDEX_NAME = "my_test_document_storage"


def _get_embedding_dim(model) -> int:
    """
    Probe qwen3_embedding_model once and cache the resulting vector
    length. Not hardcoded, since it depends on which qwen3-embedding
    tag (0.6b/4b/8b) is pulled locally — each has a different output
    dimension.
    """
    global _embedding_dim
    if _embedding_dim is None:
        _embedding_dim = len(model.embed_query("dimension probe"))
    return _embedding_dim

def _build_redis_store() -> RedisVectorStore:
    embeddings = qwen3_embedding_model
    config = RedisConfig(
        index_name=_INDEX_NAME,
        redis_url=os.getenv("REDIS_URL", "redis://localhost:6379"),
        indexing_algorithm="HNSW",
        embedding_dimensions=_get_embedding_dim(embeddings),
        from_existing=False,
    )

    return RedisVectorStore(embeddings=embeddings, config=config)
```

In [8]:

from langchain_redis import RedisVectorStore
import models.embedding_models.ollama_models
embed_model = models.embedding_models.ollama_models.qwen3_embedding_model

print(embed_model.dimensions)
vector_store = RedisVectorStore(
    models.embedding_models.ollama_models.qwen3_embedding_model,
    redis_url="redis://localhost:6379",
    index_name="my_documents"
)

None


## To fecth the dimension of the embeddng model
### this is actually a indirect way, just get a embedding of the query and get the lenght of it

In [ ]:
embedding_model =models.embedding_models.ollama_models.qwen3_embedding_model
print(embedding_model.embed_query("dimension probe"))


# Storing then embedding documents

In [ ]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Java supports virtual threads.",
        metadata={"topic": "java"}
    ),
    Document(
        page_content="Python supports list comprehensions.",
        metadata={"topic": "python"}
    ),
    Document(
        page_content="LangGraph is used for building stateful agent workflows.",
        metadata={"topic": "langgraph"}
    ),
    Document(
        page_content="LangChain provides abstractions for working with LLMs.",
        metadata={"topic": "langchain"}
    )
]
vector_store.add_documents(documents)

# Retrieve the documents using similarity search

In [ ]:
question = "What is LangGraph?"
results = vector_store.similarity_search(
    question,k=2
)

for doc in results:
    print(doc)

# Retrieve the documents using retriever

In [ ]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)


docs = retriever.invoke(question)
for doc in results:
    print(doc)

# To check the document in the  redis
## redis-cli KEYS '*'
## redis-cli SCAN 0

# Comparison between InMemory and Redis Vector store

## InMemoryVectorStore vs RedisVectorStore

| Feature | InMemoryVectorStore | RedisVectorStore |
|---|---|---|
| Storage | Python process memory | Redis |
| Persistence | ❌ No | ✅ Yes |
| Survives application restart | ❌ No | ✅ Yes |
| Shared across application instances | ❌ No | ✅ Yes |
| External infrastructure required | ❌ No | ✅ Yes |
| Vector similarity search | ✅ Yes | ✅ Yes |
| Metadata support | ✅ Yes | ✅ Yes |
| Retriever support | ✅ Yes | ✅ Yes |
| Configurable `k` | ✅ Yes | ✅ Yes |
| Filtering | Limited | ✅ Richer options |
| Scalability | Limited by application memory | Much better |
| Suitable for unit tests | ✅ Excellent | ⚠️ Usually unnecessary |
| Suitable for prototypes | ✅ Excellent | ✅ Yes |
| Suitable for production | ⚠️ Limited | ✅ Yes |
| Data shared between processes | ❌ No | ✅ Yes |
| Operational complexity | Very low | Higher |
| Speed for small datasets | Very fast | Network overhead |
| Best use case | Learning, testing, small RAG | Production/shared RAG |